# Data Quality Check

Three cells: load DB → plot per-source / per-classifier counts → list failed extractions for triage.

Run from project root with the venv kernel.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

from collector.settings import Settings

settings = Settings.load('../config.yaml')
engine = create_engine(f'sqlite:///{settings.storage.db_path}', future=True)

documents = pd.read_sql('SELECT * FROM documents', engine)
extracted = pd.read_sql('SELECT * FROM extracted_texts', engine)
tenders = pd.read_sql('SELECT * FROM tenders', engine)

print(f'tenders={len(tenders)}  documents={len(documents)}  extracted_texts={len(extracted)}')
documents.head()

In [ ]:
by_source = documents.groupby('source_id').size().sort_values(ascending=False)
by_classified = documents.groupby('classified_type').size().sort_values(ascending=False)

ax1 = by_source.plot(kind='barh', title='Documents per source')
ax1.figure.tight_layout()
ax1.figure.savefig('per_source.png')

ax2 = by_classified.plot(kind='barh', title='Documents per classified type')
ax2.figure.tight_layout()
ax2.figure.savefig('per_classified.png')

In [ ]:
failed = extracted[extracted['extraction_status'] != 'ok']
print(f'{len(failed)} failed/manual-review extractions')
failed[['document_id', 'extraction_status', 'extraction_error', 'manual_review_required']].head(20)